<a href="https://colab.research.google.com/github/OguzhanB16/Rice_type_classification/blob/main/rice_type_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Prepare dataset


In [4]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mssmartypants/rice-type-classification")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'rice-type-classification' dataset.
Path to dataset files: /kaggle/input/rice-type-classification


In [5]:
import torch
import torch.optim as optim
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchsummary import summary
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score
)

device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [6]:
data_df = pd.read_csv("/kaggle/input/rice-type-classification/riceClassification.csv")
data_df.head()

,id,Area,MajorAxisLength,MinorAxisLength,Eccentricity,ConvexArea,EquivDiameter,Extent,Perimeter,Roundness,AspectRation,Class
0,1,4537,92.229316,64.012769,0.719916,4677,76.004525,0.657536,273.085,0.764510,1.440796,1
1,2,2872,74.691881,51.400454,0.725553,3015,60.471018,0.713009,208.317,0.831658,1.453137,1
2,3,3048,76.293164,52.043491,0.731211,3132,62.296341,0.759153,210.012,0.868434,1.465950,1
3,4,3073,77.033628,51.928487,0.738639,3157,62.551300,0.783529,210.657,0.870203,1.483456,1
4,5,3693,85.124785,56.374021,0.749282,3802,68.571668,0.769375,230.332,0.874743,1.510000,1


In [7]:
data_df.dropna(inplace= True)
data_df.drop(['id'], axis= 1, inplace= True)

print(data_df.shape)

(18185, 11)


In [8]:
data_df.head()

,Area,MajorAxisLength,MinorAxisLength,Eccentricity,ConvexArea,EquivDiameter,Extent,Perimeter,Roundness,AspectRation,Class
0,4537,92.229316,64.012769,0.719916,4677,76.004525,0.657536,273.085,0.764510,1.440796,1
1,2872,74.691881,51.400454,0.725553,3015,60.471018,0.713009,208.317,0.831658,1.453137,1
2,3048,76.293164,52.043491,0.731211,3132,62.296341,0.759153,210.012,0.868434,1.465950,1
3,3073,77.033628,51.928487,0.738639,3157,62.551300,0.783529,210.657,0.870203,1.483456,1
4,3693,85.124785,56.374021,0.749282,3802,68.571668,0.769375,230.332,0.874743,1.510000,1


In [9]:
print(data_df['Class'].unique(),'\n')
print(data_df['Class'].value_counts())

[1 0] 

Class
1    9985
0    8200
Name: count, dtype: int64


In [10]:
original_df = data_df.copy()

for column in data_df.columns:
  data_df[column] = data_df[column]/data_df[column].abs().max()

data_df.head()

,Area,MajorAxisLength,MinorAxisLength,Eccentricity,ConvexArea,EquivDiameter,Extent,Perimeter,Roundness,AspectRation,Class
0,0.444368,0.503404,0.775435,0.744658,0.424873,0.666610,0.741661,0.537029,0.844997,0.368316,1.0
1,0.281293,0.407681,0.622653,0.750489,0.273892,0.530370,0.804230,0.409661,0.919215,0.371471,1.0
2,0.298531,0.416421,0.630442,0.756341,0.284520,0.546380,0.856278,0.412994,0.959862,0.374747,1.0
3,0.300979,0.420463,0.629049,0.764024,0.286791,0.548616,0.883772,0.414262,0.961818,0.379222,1.0
4,0.361704,0.464626,0.682901,0.775033,0.345385,0.601418,0.867808,0.452954,0.966836,0.386007,1.0


In [11]:
X = np.array(data_df.iloc[:, :-1])
Y = np.array(data_df.iloc[:, -1])

In [12]:
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size= 0.3)

X_train.shape, X_test.shape, y_train.shape, y_test.shape

((12729, 10), (5456, 10), (12729,), (5456,))

In [13]:
X_test, X_val, y_test, y_val = train_test_split(X_test, y_test, test_size= 0.5)

X_test.shape, X_val.shape, y_test.shape, y_val.shape

((2728, 10), (2728, 10), (2728,), (2728,))

In [14]:
class dataset(Dataset):
  def __init__(self, X, y):
    self.X = torch.tensor(X, dtype= torch.float32).to(device)
    self.y = torch.tensor(y, dtype= torch.float32).to(device)

  def __len__(self):
    return len(self.X)

  def __getitem__(self, index):
    return self.X[index], self.y[index]


train_dataset = dataset(X_train, y_train)
test_dataset = dataset(X_test, y_test)
val_dataset = dataset(X_val, y_val)

print(len(train_dataset), len(test_dataset), len(val_dataset))

12729 2728 2728


In [15]:
train_dataloader = DataLoader(train_dataset, batch_size= 32, shuffle= True)
test_dataloader = DataLoader(test_dataset, batch_size= 32, shuffle= True)
val_dataloader = DataLoader(val_dataset, batch_size= 32, shuffle= True)

## Creating a Model

In [16]:
class MyModel(nn.Module):
  def __init__(self):
    super().__init__()
    self.layer = nn.Sequential(
        nn.Linear(X.shape[1], 64),
        nn.BatchNorm1d(64),
        nn.ReLU(),
        nn.Dropout(p= 0.3),

        nn.Linear(64, 32),
        nn.BatchNorm1d(32),
        nn.ReLU(),
        nn.Dropout(p= 0.3),

        nn.Linear(32, 16),
        nn.BatchNorm1d(16),
        nn.ReLU(),
        nn.Dropout(p= 0.3),

        nn.Linear(16, 1),
    )

  def forward(self, x):
    return self.layer(x)

torch.manual_seed(42)
model_0 = MyModel().to(device)
summary(model_0, (X.shape[1],))

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Linear-1                   [-1, 64]             704
       BatchNorm1d-2                   [-1, 64]             128
              ReLU-3                   [-1, 64]               0
           Dropout-4                   [-1, 64]               0
            Linear-5                   [-1, 32]           2,080
       BatchNorm1d-6                   [-1, 32]              64
              ReLU-7                   [-1, 32]               0
           Dropout-8                   [-1, 32]               0
            Linear-9                   [-1, 16]             528
      BatchNorm1d-10                   [-1, 16]              32
             ReLU-11                   [-1, 16]               0
          Dropout-12                   [-1, 16]               0
           Linear-13                    [-1, 1]              17
Total params: 3,553
Trainable params: 3

## Train loop and Test loop

In [17]:
# Setting optimezer and loss function (nn.BCEWithLogitsLoss() because our model does not have sigmoid function)
optimizer = optim.Adam(model_0.parameters(), lr = 0.01)
loss_fn = nn.BCEWithLogitsLoss()

In [18]:
epochs = 10

for epoch in range(epochs):

    # =========================
    # TRAIN
    # =========================

    model_0.train()

    total_loss_train = 0
    correct_train = 0
    total_train = 0

    for X, y in train_dataloader:

        # Forward
        y_pred = model_0(X).squeeze(1)

        # Loss
        loss = loss_fn(y_pred, y)

        # Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Loss
        total_loss_train += loss.item()

        # Prediction
        probs = torch.sigmoid(y_pred)
        preds = (probs >= 0.5).float()

        # Accuracy
        correct_train += (preds == y).sum().item()
        total_train += y.size(0)

    train_loss = total_loss_train / len(train_dataloader)
    train_acc = correct_train / total_train


    # =========================
    # VALIDATION
    # =========================

    model_0.eval()

    total_loss_val = 0
    correct_val = 0
    total_val = 0

    with torch.inference_mode():

        for X, y in val_dataloader:

            # Forward
            y_pred = model_0(X).squeeze(1)

            # Loss
            loss = loss_fn(y_pred, y)

            total_loss_val += loss.item()

            # Prediction
            probs = torch.sigmoid(y_pred)
            preds = (probs >= 0.5).float()

            # Accuracy
            correct_val += (preds == y).sum().item()
            total_val += y.size(0)

    val_loss = total_loss_val / len(val_dataloader)
    val_acc = correct_val / total_val


    # =========================
    # PRINT
    # =========================

    print(
        f"Epoch: {epoch + 1} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.4f}"
    )


# ==================================================
# TEST
# ==================================================

model_0.eval()

all_y = []
all_probs = []
all_preds = []

with torch.inference_mode():

    for X, y in test_dataloader:

        # Model output
        logits = model_0(X).squeeze(1)

        # Probability
        probs = torch.sigmoid(logits)

        # Class prediction
        preds = (probs >= 0.5).float()

        # Save results
        all_y.extend(y.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())


# ==================================================
# METRICS
# ==================================================

accuracy = accuracy_score(all_y, all_preds)

precision = precision_score(all_y, all_preds)

recall = recall_score(all_y, all_preds)

f1 = f1_score(all_y, all_preds)

auc = roc_auc_score(all_y, all_probs)

cm = confusion_matrix(all_y, all_preds)


print("\n================ TEST RESULTS ================")

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"ROC-AUC  : {auc:.4f}")

print("\nConfusion Matrix:")
print(cm)

Epoch: 1 | Train Loss: 0.0910 | Train Acc: 0.9752 | Val Loss: 0.0466 | Val Acc: 0.9864
Epoch: 2 | Train Loss: 0.0619 | Train Acc: 0.9808 | Val Loss: 0.0345 | Val Acc: 0.9905
Epoch: 3 | Train Loss: 0.0589 | Train Acc: 0.9804 | Val Loss: 0.0886 | Val Acc: 0.9589
Epoch: 4 | Train Loss: 0.0582 | Train Acc: 0.9801 | Val Loss: 0.0374 | Val Acc: 0.9879
Epoch: 5 | Train Loss: 0.0546 | Train Acc: 0.9822 | Val Loss: 0.0344 | Val Acc: 0.9897
Epoch: 6 | Train Loss: 0.0552 | Train Acc: 0.9836 | Val Loss: 0.0632 | Val Acc: 0.9780
Epoch: 7 | Train Loss: 0.0481 | Train Acc: 0.9859 | Val Loss: 0.0382 | Val Acc: 0.9872
Epoch: 8 | Train Loss: 0.0511 | Train Acc: 0.9837 | Val Loss: 0.0681 | Val Acc: 0.9857
Epoch: 9 | Train Loss: 0.0497 | Train Acc: 0.9852 | Val Loss: 0.0332 | Val Acc: 0.9894
Epoch: 10 | Train Loss: 0.0477 | Train Acc: 0.9859 | Val Loss: 0.0394 | Val Acc: 0.9894

================ TEST RESULTS ================
Accuracy : 0.9879
Precision: 0.9809
Recall   : 0.9973
F1 Score : 0.9890
ROC-AUC  